In [60]:
import requests
import json

In [108]:
id = "dofzd80wtspaf5"

In [109]:
# xxxxxxxxx-8000
url_id = f"https://{id}-8000.proxy.runpod.net/"

In [110]:
# 헬스 체크
health_check_url = f"{url_id}health"

In [111]:
health_check_res = requests.get(health_check_url)
if health_check_res.status_code != 200:
    print("헬스 체크 실패")
    exit()

print(health_check_res.json())


{'status': 'ok'}


In [112]:
# 모델 목록 
model_list_url = f"{url_id}v1/models"

In [113]:
model_list_res = requests.get(model_list_url)

if model_list_res.status_code != 200:
    print("모델 목록 조회 실패")
    exit()

model_list = model_list_res.json()

print(model_list)

# model_name = model_list['data'][0]['id']

# print(model_name)
model_index = []

for model in model_list['data']:
    print('-'*100)
    print(model['id'])
    print('-'*100)
    model_index.append(model['id'])
    

{'object': 'list', 'data': [{'id': 'nvidia/Llama-4-Scout-17B-16E-Instruct-FP8', 'object': 'model', 'created': 1776838885, 'owned_by': 'vllm', 'root': 'nvidia/Llama-4-Scout-17B-16E-Instruct-FP8', 'parent': None, 'max_model_len': 32768, 'permission': [{'id': 'modelperm-81c64b4a08399768', 'object': 'model_permission', 'created': 1776838885, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}
----------------------------------------------------------------------------------------------------
nvidia/Llama-4-Scout-17B-16E-Instruct-FP8
----------------------------------------------------------------------------------------------------


In [114]:

url = f"{url_id}v1/chat/completions"

In [100]:
payload = {
    "model": model_index[0],
    "messages": [
        {"role": "user", "content": "딸기 제철 시기는 언제야?"}
    ],
    "max_tokens": 4000,
    "temperature": 0,
    "stream": False
}

In [101]:
res = requests.post(url, json=payload)

In [102]:
print(res)


<Response [200]>


In [103]:
print(res.json())

{'id': 'chatcmpl-80e09bc1b2645cc8', 'object': 'chat.completion', 'created': 1776837080, 'model': 'RedHatAI/Llama-4-Scout-17B-16E-Instruct-FP8-dynamic', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '딸기는 1월부터 6월까지 제철인 과일입니다. \n\n국내 딸기 생산은 1월부터 시작해 7월 중순까지 이뤄집니다. \n\n수출은 8월 말부터 시작합니다. \n\n제철이 시작되면 더욱 맛있는 딸기를 만나실 수 있을 겁니다.', 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning': None}, 'logprobs': None, 'finish_reason': 'stop', 'stop_reason': None, 'token_ids': None}], 'service_tier': None, 'system_fingerprint': None, 'usage': {'prompt_tokens': 20, 'total_tokens': 87, 'completion_tokens': 67, 'prompt_tokens_details': None}, 'prompt_logprobs': None, 'prompt_token_ids': None, 'kv_transfer_params': None}


In [115]:
payload = {
    "model": model_index[0],
    "messages": [
        {"role": "user", "content": "html, css ,js 로 테트리스 게임을 만들어줘"}
    ],
    "temperature": 0,
    "stream": True,
    # qwen3 용
    "chat_template_kwargs": {"enable_thinking": False}
}

In [26]:
res = requests.post(url, json=payload)

for line in res.iter_lines():
    if line:
        print(line.decode('utf-8'))

data: {"id":"chatcmpl-bd7475b05baaa19c","object":"chat.completion.chunk","created":1776738400,"model":"XiaomiMiMo/MiMo-7B-RL-0530","choices":[{"index":0,"delta":{"role":"assistant","content":""},"logprobs":null,"finish_reason":null}],"prompt_token_ids":null}
data: {"id":"chatcmpl-bd7475b05baaa19c","object":"chat.completion.chunk","created":1776738400,"model":"XiaomiMiMo/MiMo-7B-RL-0530","choices":[{"index":0,"delta":{"content":"<th"},"logprobs":null,"finish_reason":null,"token_ids":null}]}
data: {"id":"chatcmpl-bd7475b05baaa19c","object":"chat.completion.chunk","created":1776738400,"model":"XiaomiMiMo/MiMo-7B-RL-0530","choices":[{"index":0,"delta":{"content":"ink"},"logprobs":null,"finish_reason":null,"token_ids":null}]}
data: {"id":"chatcmpl-bd7475b05baaa19c","object":"chat.completion.chunk","created":1776738400,"model":"XiaomiMiMo/MiMo-7B-RL-0530","choices":[{"index":0,"delta":{"content":">"},"logprobs":null,"finish_reason":null,"token_ids":null}]}
data: {"id":"chatcmpl-bd7475b05baaa

In [116]:
res = requests.post(url, json=payload)

is_reasoning = False

for line in res.iter_lines():
    if line:
        decoded = line.decode('utf-8')

        if decoded.startswith('data: '):
            data = decoded.split('data: ')[1]

            if data == '[DONE]':
                break;
            
            json_data = json.loads(data)
            
            tokens = json_data['choices'][0]['delta']

            if tokens.get('reasoning', None):
                if not is_reasoning: 
                    print("<REASONING>")
                    is_reasoning = True
                print(tokens['reasoning'], end='', flush=True)
            
            if tokens.get('content', None):
                if is_reasoning:
                    print("\n<RESPONSE>")
                    is_reasoning = False
                print(tokens['content'], end='', flush=True)

Here is a basic implementation of a Tetris game using HTML, CSS, and JavaScript.

**index.html**
```html
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Tetris</title>
    <link rel="stylesheet" href="style.css">
</head>
<body>
    <div class="game-container">
        <canvas id="game-canvas" width="300" height="600"></canvas>
        <div class="score-container">
            <span id="score">0</span>
        </div>
    </div>
    <script src="script.js"></script>
</body>
</html>
```

**style.css**
```css
.game-container {
    display: flex;
    flex-direction: column;
    align-items: center;
}

#game-canvas {
    border: 1px solid black;
}

.score-container {
    font-size: 24px;
    font-weight: bold;
    margin-top: 20px;
}
```

**script.js**
```javascript
// 게임 설정
const GAME_WIDTH = 10;
const GAME_HEIGHT = 20;
const BLOCK_SIZE = 30;

// 블록 종류
const BLOCK_TYPES = [
    // I 블록
 